In [ ]:
#CUT3R depth map viewer 
import numpy as np
import matplotlib.pyplot as plt

arr = np.load(r'F:/MScPROJECT_LOCAL/Data/07_tropea_cafe/035_CUT3R_output/depth/000007.npy')
plt.imshow(arr)
plt.colorbar()
plt.show()

In [ ]:
#XYZ positions (subject) 
import matplotlib.pyplot as plt
import numpy as np

keys_ra1 = sorted(subject_positions_model_dict.keys())
keys_ra8 = sorted(subject_positions_model_dict_RA8.keys())

fig, axes = plt.subplots(3, 1, sharex=True, figsize=(8, 6))
for i, (ax, label) in enumerate(zip(axes, ("x", "y", "z"))):
    vals_ra1 = np.array([subject_positions_model_dict[k][i] for k in keys_ra1])
    vals_ra8 = np.array([subject_positions_model_dict_RA8[k][i] for k in keys_ra8])
    spread_ra8 = depth_spread_model_RA8[:, i]

    ax.fill_between(keys_ra8, vals_ra8 - spread_ra8, vals_ra8 + spread_ra8, color="0.7", alpha=0.6, label="RA8 spread")
    ax.plot(keys_ra1, vals_ra1, marker="o", color="red", markersize=1, alpha=0.6, label="RA1")
    ax.plot(keys_ra8, vals_ra8, marker="o", color="grey", markersize=1, alpha = 0.8, label="RA8")
    ax.set_ylabel(label)

axes[0].legend()
axes[-1].set_xlabel("Frame")

fig.tight_layout()


In [ ]:
# frame test  counting- 5 min video lets see
prompt = "return the number you see every frame you check in a simple list"
import os
from google import genai

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

myfile = client.files.upload(file="/workspace/project/Data/control_frame_index.mp4")

while myfile.state.name != "ACTIVE":
    import time; time.sleep(5)
    myfile = client.files.get(name=myfile.name)

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=[myfile, prompt],
)
print(response.text)
import re
from matplotlib import pyplot as plt
numbers = [int(n) for n in re.findall(r'-?\d+', response.text)]
plt.plot(numbers)
diffs = [b - a for a, b in zip(numbers, numbers[1:])]
print(diffs)
plt.plot(diffs)
drop_idxs = [i for i, v in enumerate(diffs) if v != 30]
intervals = [b - a for a, b in zip(drop_idxs, drop_idxs[1:])]
print(intervals)
drop_idxs = [i for i, v in enumerate(diffs) if v != 30]
frames_between_drops = [sum(diffs[a:b]) for a, b in zip(drop_idxs, drop_idxs[1:])]
print(frames_between_drops)
fps = 30000 / 1001
interval_s = [f / fps for f in frames_between_drops]
print(interval_s)

